# Data Parallel Comparison: DDP vs FSDP vs ZeRO

## Overview

This notebook provides a comprehensive comparison of data parallelism strategies with benchmarks and decision guidelines.

### Topics Covered
- Memory footprint comparison
- Communication overhead analysis
- Throughput benchmarks
- Decision framework

## 1. Memory Comparison

### Memory Formula Summary

For model with $\Psi$ parameters, $N$ GPUs, mixed precision (FP16 + FP32 optimizer):

| Method | Memory per GPU | Formula |
|--------|---------------|----------|
| DDP | $16\Psi$ | Full replication |
| ZeRO-1 | $4\Psi + 12\Psi/N$ | Shard optimizer |
| ZeRO-2 | $2\Psi + 14\Psi/N$ | + Shard gradients |
| ZeRO-3/FSDP | $16\Psi/N$ | Shard everything |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def memory_comparison(num_params_b, num_gpus_list=[1, 2, 4, 8, 16, 32]):
    """Compare memory usage across methods."""
    psi = num_params_b * 1e9  # Convert to actual params
    
    results = {'GPUs': num_gpus_list}
    
    for N in num_gpus_list:
        ddp = 16 * psi / 1e9  # GB
        zero1 = (4 * psi + 12 * psi / N) / 1e9
        zero2 = (2 * psi + 14 * psi / N) / 1e9
        zero3 = 16 * psi / N / 1e9
        
    print(f"Model: {num_params_b}B parameters")
    print(f"\n{'GPUs':<6} {'DDP':<12} {'ZeRO-1':<12} {'ZeRO-2':<12} {'ZeRO-3':<12}")
    print("-" * 54)
    
    for N in num_gpus_list:
        ddp = 16 * psi / 1e9
        zero1 = (4 * psi + 12 * psi / N) / 1e9
        zero2 = (2 * psi + 14 * psi / N) / 1e9
        zero3 = 16 * psi / N / 1e9
        print(f"{N:<6} {ddp:<12.1f} {zero1:<12.1f} {zero2:<12.1f} {zero3:<12.1f}")

memory_comparison(7)  # 7B model

## 2. Communication Overhead

### Communication Volume per Step

| Method | Forward | Backward | Total |
|--------|---------|----------|-------|
| DDP | 0 | $2\Psi$ (AllReduce) | $2\Psi$ |
| ZeRO-1 | 0 | $2\Psi$ (AllReduce) | $2\Psi$ |
| ZeRO-2 | 0 | $2\Psi$ (ReduceScatter) | $2\Psi$ |
| ZeRO-3 | $\Psi$ (AllGather) | $2\Psi$ (AG + RS) | $3\Psi$ |

In [ ]:
def communication_analysis(num_params_b, bandwidth_gbps=100):
    """Analyze communication time for different methods."""
    psi_bytes = num_params_b * 1e9 * 2  # FP16 = 2 bytes
    bandwidth_bytes = bandwidth_gbps * 1e9 / 8  # Convert to bytes/s
    
    ddp_time = 2 * psi_bytes / bandwidth_bytes * 1000  # ms
    zero3_time = 3 * psi_bytes / bandwidth_bytes * 1000
    
    print(f"Model: {num_params_b}B params, Bandwidth: {bandwidth_gbps} Gbps")
    print(f"\nCommunication time per step:")
    print(f"  DDP/ZeRO-1/2: {ddp_time:.1f} ms")
    print(f"  ZeRO-3/FSDP:  {zero3_time:.1f} ms")
    print(f"  Overhead:     {(zero3_time/ddp_time - 1)*100:.0f}%")

communication_analysis(7, 100)  # 7B model, 100 Gbps network

## 3. Decision Framework

```
Model fits in single GPU memory?
├── YES → Use DDP (fastest)
└── NO → How much does it exceed?
         ├── Slightly (1.5-2x) → ZeRO-2 or FSDP SHARD_GRAD_OP
         ├── Moderately (2-4x) → ZeRO-3 or FSDP FULL_SHARD
         └── Significantly (>4x) → ZeRO-3 + Offload or Tensor Parallel
```

### Quick Reference

| Model Size | GPUs | Recommendation |
|------------|------|----------------|
| < 1B | 1-8 | DDP |
| 1-7B | 4-16 | ZeRO-2 / FSDP |
| 7-30B | 8-64 | ZeRO-3 / FSDP |
| 30-100B | 32-256 | ZeRO-3 + TP |
| > 100B | 256+ | 3D Parallelism |

## 4. Summary

### Key Takeaways

1. **DDP**: Best throughput when model fits in memory
2. **ZeRO-1/2**: Good balance of memory savings and speed
3. **ZeRO-3/FSDP**: Maximum memory efficiency, ~50% more communication
4. **Offload**: Last resort for extreme memory constraints